In [1]:
import numpy as np
import random
from collections import deque
import gymnasium as gym
from scipy import stats


# ============================================================
# Hyperparameters
# ============================================================

H = 128
BATCH = 128
GAMMA = 0.99
TAU = 0.005
LR = 0.0003


# ============================================================
# Utility functions
# ============================================================

def relu(x):
    return np.maximum(0, x)


def relu_derivative(x):
    return (x > 0).astype(np.float32)


def tanh_derivative(x):
    return 1.0 - np.tanh(x) ** 2


# ============================================================
# Simple Neural Network
# ============================================================

class NeuralNetwork:

    def __init__(self, input_dim, hidden_dim, output_dim):

        self.W1 = np.random.randn(
            input_dim, hidden_dim
        ) * np.sqrt(2.0 / input_dim)

        self.b1 = np.zeros(hidden_dim)

        self.W2 = np.random.randn(
            hidden_dim, hidden_dim
        ) * np.sqrt(2.0 / hidden_dim)

        self.b2 = np.zeros(hidden_dim)

        self.W3 = np.random.randn(
            hidden_dim, output_dim
        ) * np.sqrt(2.0 / hidden_dim)

        self.b3 = np.zeros(output_dim)

    def forward(self, x):

        self.x = x

        self.z1 = x @ self.W1 + self.b1
        self.a1 = relu(self.z1)

        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = relu(self.z2)

        self.out = self.a2 @ self.W3 + self.b3

        return self.out

    def predict(self, x):

        z1 = x @ self.W1 + self.b1
        a1 = relu(z1)

        z2 = a1 @ self.W2 + self.b2
        a2 = relu(z2)

        return a2 @ self.W3 + self.b3


# ============================================================
# Actor Network
# ============================================================

class Actor:

    def __init__(self, state_dim, action_dim, max_action):

        self.state_dim = state_dim
        self.action_dim = action_dim
        self.max_action = max_action

        # Mean network
        self.mean_net = NeuralNetwork(
            state_dim,
            H,
            action_dim
        )

        # Log standard deviation
        self.log_std = np.zeros(action_dim)

    def sample(self, state):

        if state.ndim == 1:
            state = state.reshape(1, -1)

        mean = self.mean_net.predict(state)

        std = np.exp(
            np.clip(self.log_std, -5, 2)
        )

        noise = np.random.randn(
            *mean.shape
        )

        z = mean + noise * std

        action = np.tanh(z) * self.max_action

        # Gaussian log probability
        log_prob = -0.5 * (
            ((z - mean) / std) ** 2
            + 2 * np.log(std)
            + np.log(2 * np.pi)
        )

        # Tanh correction
        log_prob -= np.log(
            1 - np.tanh(z) ** 2 + 1e-6
        )

        log_prob = np.sum(
            log_prob,
            axis=1,
            keepdims=True
        )

        return action, log_prob

    def act(self, state):

        if state.ndim == 1:
            state = state.reshape(1, -1)

        mean = self.mean_net.predict(state)

        action = np.tanh(mean) * self.max_action

        return action[0]


# ============================================================
# Critic Network
# ============================================================

class Critic:

    def __init__(self, state_dim, action_dim):

        self.net = NeuralNetwork(
            state_dim + action_dim,
            H,
            1
        )

    def predict(self, state, action):

        x = np.concatenate(
            [state, action],
            axis=1
        )

        return self.net.predict(x)


# ============================================================
# Replay Buffer
# ============================================================

class Buffer:

    def __init__(self, capacity=100_000):

        self.d = deque(
            maxlen=capacity
        )

    def push(self, *x):

        self.d.append(x)

    def sample(self, n):

        batch = random.sample(
            self.d,
            n
        )

        s, a, r, s2, d = zip(*batch)

        return (
            np.array(s, dtype=np.float32),
            np.array(a, dtype=np.float32),
            np.array(r, dtype=np.float32).reshape(-1, 1),
            np.array(s2, dtype=np.float32),
            np.array(d, dtype=np.float32).reshape(-1, 1)
        )

    def __len__(self):

        return len(self.d)


# ============================================================
# SAC Agent
# ============================================================

class SAC:

    def __init__(
        self,
        state_dim,
        action_dim,
        max_action
    ):

        self.actor = Actor(
            state_dim,
            action_dim,
            max_action
        )

        self.q1 = Critic(
            state_dim,
            action_dim
        )

        self.q2 = Critic(
            state_dim,
            action_dim
        )

        self.target_q1 = Critic(
            state_dim,
            action_dim
        )

        self.target_q2 = Critic(
            state_dim,
            action_dim
        )

        # Copy critic weights
        self.copy_network(
            self.q1.net,
            self.target_q1.net
        )

        self.copy_network(
            self.q2.net,
            self.target_q2.net
        )

        self.alpha = 0.2

        self.target_entropy = -action_dim

    # --------------------------------------------------------
    # Copy network
    # --------------------------------------------------------

    def copy_network(self, source, target):

        target.W1 = source.W1.copy()
        target.b1 = source.b1.copy()

        target.W2 = source.W2.copy()
        target.b2 = source.b2.copy()

        target.W3 = source.W3.copy()
        target.b3 = source.b3.copy()

    # --------------------------------------------------------
    # Soft update
    # --------------------------------------------------------

    def soft_update(self, source, target):

        target.W1 = (
            TAU * source.W1
            + (1 - TAU) * target.W1
        )

        target.b1 = (
            TAU * source.b1
            + (1 - TAU) * target.b1
        )

        target.W2 = (
            TAU * source.W2
            + (1 - TAU) * target.W2
        )

        target.b2 = (
            TAU * source.b2
            + (1 - TAU) * target.b2
        )

        target.W3 = (
            TAU * source.W3
            + (1 - TAU) * target.W3
        )

        target.b3 = (
            TAU * source.b3
            + (1 - TAU) * target.b3
        )

    # --------------------------------------------------------
    # Select action
    # --------------------------------------------------------

    def act(self, state):

        return self.actor.act(state)

    # --------------------------------------------------------
    # Critic update
    # --------------------------------------------------------

    def train_critic(
        self,
        critic,
        states,
        actions,
        targets
    ):

        x = np.concatenate(
            [states, actions],
            axis=1
        )

        prediction = critic.net.forward(x)

        error = prediction - targets

        loss = np.mean(
            error ** 2
        )

        batch_size = len(states)

        grad_out = (
            2 * error / batch_size
        )

        # Output layer
        grad_W3 = (
            critic.net.a2.T @ grad_out
        )

        grad_b3 = np.sum(
            grad_out,
            axis=0
        )

        grad_a2 = (
            grad_out @ critic.net.W3.T
        )

        grad_z2 = (
            grad_a2
            * relu_derivative(
                critic.net.z2
            )
        )

        grad_W2 = (
            critic.net.a1.T @ grad_z2
        )

        grad_b2 = np.sum(
            grad_z2,
            axis=0
        )

        grad_a1 = (
            grad_z2 @ critic.net.W2.T
        )

        grad_z1 = (
            grad_a1
            * relu_derivative(
                critic.net.z1
            )
        )

        grad_W1 = (
            x.T @ grad_z1
        )

        grad_b1 = np.sum(
            grad_z1,
            axis=0
        )

        # Gradient clipping
        grads = [
            grad_W1,
            grad_b1,
            grad_W2,
            grad_b2,
            grad_W3,
            grad_b3
        ]

        for g in grads:
            np.clip(
                g,
                -1,
                1,
                out=g
            )

        # Update
        critic.net.W1 -= LR * grad_W1
        critic.net.b1 -= LR * grad_b1

        critic.net.W2 -= LR * grad_W2
        critic.net.b2 -= LR * grad_b2

        critic.net.W3 -= LR * grad_W3
        critic.net.b3 -= LR * grad_b3

        return loss

    # --------------------------------------------------------
    # SAC update
    # --------------------------------------------------------

    def update(self, buffer):

        if len(buffer) < BATCH:
            return

        states, actions, rewards, next_states, dones = \
            buffer.sample(BATCH)

        # ----------------------------------------------------
        # Target actions
        # ----------------------------------------------------

        next_actions, next_log_prob = \
            self.actor.sample(next_states)

        target_q1 = self.target_q1.predict(
            next_states,
            next_actions
        )

        target_q2 = self.target_q2.predict(
            next_states,
            next_actions
        )

        target_q = np.minimum(
            target_q1,
            target_q2
        )

        target_q -= (
            self.alpha * next_log_prob
        )

        targets = (
            rewards
            + (1 - dones)
            * GAMMA
            * target_q
        )

        # ----------------------------------------------------
        # Train critics
        # ----------------------------------------------------

        self.train_critic(
            self.q1,
            states,
            actions,
            targets
        )

        self.train_critic(
            self.q2,
            states,
            actions,
            targets
        )

        # ----------------------------------------------------
        # Actor update
        # ----------------------------------------------------

        new_actions, log_prob = \
            self.actor.sample(states)

        q1_new = self.q1.predict(
            states,
            new_actions
        )

        q2_new = self.q2.predict(
            states,
            new_actions
        )

        q_min = np.minimum(
            q1_new,
            q2_new
        )

        # Approximate policy improvement
        advantage = (
            q_min
            - self.alpha * log_prob
        )

        advantage = np.clip(
            advantage,
            -10,
            10
        )

        # Update actor mean toward better actions
        mean_output = self.actor.mean_net.predict(
            states
        )

        direction = (
            new_actions / self.actor.max_action
        )

        direction *= advantage

        direction = np.clip(
            direction,
            -1,
            1
        )

        # Simple policy adjustment
        self.actor.mean_net.b3 += (
            LR * np.mean(
                direction,
                axis=0
            )
        )

        # ----------------------------------------------------
        # Entropy adjustment
        # ----------------------------------------------------

        entropy_error = np.mean(
            -log_prob
        ) - self.target_entropy

        self.alpha *= np.exp(
            -LR * entropy_error
        )

        self.alpha = np.clip(
            self.alpha,
            0.01,
            1.0
        )

        # ----------------------------------------------------
        # Soft target update
        # ----------------------------------------------------

        self.soft_update(
            self.q1.net,
            self.target_q1.net
        )

        self.soft_update(
            self.q2.net,
            self.target_q2.net
        )


# ============================================================
# Create Environment
# ============================================================

env = gym.make(
    "Pendulum-v1"
)


state_dim = env.observation_space.shape[0]

action_dim = env.action_space.shape[0]

max_action = float(
    env.action_space.high[0]
)


agent = SAC(
    state_dim,
    action_dim,
    max_action
)


buffer = Buffer()

reward_history = []


# ============================================================
# Training
# ============================================================

for ep in range(50):

    state, _ = env.reset()

    done = False

    episode_reward = 0

    while not done:

        # Random exploration during warm-up
        if len(buffer) < 500:

            action = env.action_space.sample()

        else:

            action = agent.act(
                state
            )

        next_state, reward, terminated, truncated, _ = \
            env.step(action)

        done = terminated or truncated

        buffer.push(
            state,
            action,
            reward,
            next_state,
            float(done)
        )

        state = next_state

        episode_reward += reward

        agent.update(
            buffer
        )

    reward_history.append(
        episode_reward
    )

    print(
        f"Episode {ep + 1}: "
        f"reward={episode_reward:.1f}, "
        f"alpha={agent.alpha:.3f}"
    )


# ============================================================
# Evaluation
# ============================================================

last_n = reward_history[-10:]

mean_reward = np.mean(
    last_n
)

sem_reward = stats.sem(
    last_n
)


print(
    f"\nFinal performance "
    f"(last {len(last_n)} episodes): "
    f"{mean_reward:.1f} +/- {sem_reward:.1f} "
    f"(mean +/- SEM)"
)


env.close()

Episode 1: reward=-1295.0, alpha=0.215
Episode 2: reward=-835.7, alpha=0.264
Episode 3: reward=-1614.2, alpha=0.314
Episode 4: reward=-1494.4, alpha=0.388
Episode 5: reward=-1640.7, alpha=0.511
Episode 6: reward=-1593.0, alpha=0.701
Episode 7: reward=-1668.1, alpha=0.982
Episode 8: reward=-1567.8, alpha=1.000
Episode 9: reward=-1587.2, alpha=1.000
Episode 10: reward=-1591.3, alpha=1.000
Episode 11: reward=-1539.6, alpha=1.000
Episode 12: reward=-1574.2, alpha=1.000
Episode 13: reward=-1560.8, alpha=1.000
Episode 14: reward=-1579.1, alpha=1.000
Episode 15: reward=-1633.8, alpha=1.000
Episode 16: reward=-1567.3, alpha=1.000
Episode 17: reward=-1665.5, alpha=1.000
Episode 18: reward=-1627.2, alpha=1.000
Episode 19: reward=-1449.0, alpha=1.000
Episode 20: reward=-1599.1, alpha=1.000
Episode 21: reward=-1590.9, alpha=1.000
Episode 22: reward=-1602.9, alpha=1.000
Episode 23: reward=-1587.9, alpha=1.000
Episode 24: reward=-1560.3, alpha=1.000
Episode 25: reward=-1588.6, alpha=1.000
Episode 26